# Load model checkpoint and evaluate performance

In [1]:
import torch
import sys
sys.path.append("../")
sys.path.append("../src")
sys.path.append("../src/models")
sys.path.append("../src/data")
sys.path.append("../src/utils")
sys.path.append("../src/data/components/")
sys.path.append("../src/models/components/")
sys.path.append("../src/utils/IEBCS")
sys.path.append("../src/utils/IEBCS/representations")
import eventIO, event_representations
from topspin_datamodule import TopspinDataModule
from topspin_classification_module import TopspinLitModule
from tqdm import tqdm
import numpy as np

/home/lkolmar/anaconda3/envs/learning/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checpoint_path = "../logs/train/runs/" + "2025-08-08_15-40-29/" + "checkpoints/" + "epoch_034.ckpt"

# model = torch.load(checpoint_path, map_location=torch.device('cpu'), weights_only=False)
model = TopspinLitModule.load_from_checkpoint(checpoint_path)

/home/lkolmar/anaconda3/envs/learning/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'net' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['net'])`.


In [7]:
data_module = TopspinDataModule(
    data_dir="/data/lkolmar/datasets/topspin_fit_to_max/",
    time_window=5000,  # in us
    num_bins=10,
    sensor_size=(100, 100),
    train_val_test_split=(1294, 277, 277),  # n = 1848 (70%, 15%, 15%)
    batch_size=8,
    num_workers=4,
    pin_memory=True,
)
data_module.prepare_data()
data_module.setup()
test_loader = data_module.test_dataloader()

[229, 1637, 1827, 315, 1373, 1100, 810, 160, 1573, 864, 1516, 837, 586, 341, 361, 527, 881, 1125, 158, 1677, 874, 1245, 1301, 503, 1508, 995, 99, 19, 278, 76, 851, 919, 342, 1383, 127, 1759, 927, 88, 1834, 1498, 556, 129, 462, 1585, 1078, 911, 293, 114, 1267, 943, 214, 526, 1595, 196, 5, 330, 299, 716, 679, 331, 1711, 733, 372, 126, 1461, 703, 77, 474, 1415, 647, 1471, 1786, 45, 494, 153, 1094, 696, 30, 1172, 833, 1625, 725, 1111, 1420, 1375, 657, 1056, 879, 452, 961, 547, 1281, 1543, 1241, 1806, 230, 403, 173, 1254, 1787, 1028, 681, 117, 304, 1636, 1050, 314, 869, 1752, 699, 710, 1258, 1164, 258, 896, 1190, 1351, 287, 797, 1432, 1746, 1230, 1080, 496, 1644, 1613, 1413, 9, 379, 780, 1463, 1754, 410, 565, 337, 1322, 356, 1359, 206, 40, 236, 1087, 1483, 1137, 905, 928, 1119, 1833, 20, 1314, 1043, 1047, 621, 1280, 708, 845, 1591, 415, 1841, 618, 1386, 268, 1151, 251, 913, 14, 652, 409, 885, 1112, 1157, 844, 1737, 1757, 557, 327, 429, 1428, 1382, 324, 328, 920, 693, 1072, 904, 1820, 581, 4

In [4]:
labels = {
    "topspin_slow": 0,
    "topspin_mid": 1,
    "topspin_fast": 2,
    "backspin_slow": 3,
    "backspin_mid": 4,
    "backspin_fast": 5,
}

In [5]:
print("Number of test samples:", len(test_loader.dataset))
samples_per_class = [0, 0, 0, 0, 0, 0]
for c in range(len(test_loader.dataset)):
    label = test_loader.dataset[c][1]
    samples_per_class[label] += 1

print("Samples per class:", samples_per_class)

Number of test samples: 277
Samples per class: [6, 32, 99, 8, 36, 96]


In [12]:
# [index, label, prediction]
predictions = []
model.eval()
model.to("cpu")

with torch.no_grad():
    for batch in test_loader:
        print(len(batch))
        # print(batch)

        logits = model.net(batch[0].to("cpu"), batch[1].to("cpu"))
        logits = np.argmax(logits.cpu().numpy(), axis=1)

        predictions.extend(zip(logits, batch[1]))

print(predictions[:10])  # Print first 10 predictions

3
3


KeyboardInterrupt: 